In [1]:
# 必要なら（既に入っていれば不要）
# %pip install -q datasets torch

import re
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset


# ----------------------------
# データ読み込み（Hugging Face datasets）
# ----------------------------
ds = load_dataset("ag_news")  # ds["train"], ds["test"] が使える

class AGNewsHFDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # HFのag_newsは label が 0..3（torchtextと違って 1..4 じゃない）
        return int(item["label"]), item["text"]


train_ds = AGNewsHFDataset(ds["train"])
test_ds  = AGNewsHFDataset(ds["test"])


# ----------------------------
# tokenizer（torchtextの basic_english っぽい簡易版）
# ----------------------------
def basic_english_tokenizer(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # 記号を空白に
    text = re.sub(r"\s+", " ", text).strip()   # 連続空白を詰める
    return text.split()


# ----------------------------
# vocab（torchtextなしで自作）
# ----------------------------
class Vocab:
    def __init__(self, stoi, itos, default_index: int):
        self.stoi = stoi
        self.itos = itos
        self.default_index = default_index

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token: str):
        return self.stoi.get(token, self.default_index)

    def __call__(self, tokens):
        return [self[token] for token in tokens]


def build_vocab_from_texts(text_iter, tokenizer, specials=("<unk>", "<pad>"), min_freq=2):
    counter = Counter()
    for text in text_iter:
        toks = tokenizer(text)
        if not toks:
            toks = ["<unk>"]
        counter.update(toks)

    itos = list(specials)
    # 安定するように freq desc → token asc で並べる
    for tok, freq in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        if freq >= min_freq and tok not in specials:
            itos.append(tok)

    stoi = {tok: i for i, tok in enumerate(itos)}
    default_index = stoi["<unk>"]
    return Vocab(stoi=stoi, itos=itos, default_index=default_index)


# train から vocab 作成
vocab = build_vocab_from_texts(
    (ds["train"][i]["text"] for i in range(len(ds["train"]))),
    tokenizer=basic_english_tokenizer,
    specials=("<unk>", "<pad>"),
    min_freq=2
)
pad_idx = vocab["<pad>"]


def text_pipeline(x: str):
    tokens = basic_english_tokenizer(x)
    if not tokens:
        tokens = ["<unk>"]
    return vocab(tokens)

def label_pipeline(y: int):
    # HFのag_newsは 0..3 のままでOK
    return int(y)


# ----------------------------
# DataLoader 用 collate（padding + lengths）
# ----------------------------
def collate_batch(batch):
    labels = []
    sequences = []
    lengths = []

    for (label, text) in batch:
        labels.append(label_pipeline(label))
        ids = torch.tensor(text_pipeline(text), dtype=torch.long)
        sequences.append(ids)
        lengths.append(ids.size(0))

    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    tokens_padded = pad_sequence(sequences, batch_first=True, padding_value=pad_idx)
    return tokens_padded, lengths, labels


batch_size = 64
trainloader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch, num_workers=0)
testloader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_batch, num_workers=0)




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Nutzer\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Nutzer\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Nutzer\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\Nutzer\anaconda3\Lib\site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Nutzer\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Nutzer\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Nutzer\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\Nutzer\anaconda3\Lib\site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [3]:
import torch

In [2]:
# BiRNN（双方向RNN）で AG_NEWS（文章分類：4クラス）を学習する “完全版” サンプル
# 目的：CIFAR10みたいに「Dataset → DataLoader → Net → 学習 → 評価」をRNNで再現

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchtext.datasets import AG_NEWS
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


# ----------------------------
# モデル定義（BiRNN）
# __init__：層の構成
# forward：データの流れ
# ----------------------------
class BiRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_layers=1, num_classes=4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # bidirectional=True -> 方向が2つ（forward/backward）
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        # 本の双方向出力の合成（前向き + 後ろ向き）に合わせて hidden_size -> num_classes
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        """
        tokens:  (N, T)  パディング済みの単語ID
        lengths: (N,)    各文の長さ（パディング前）
        return:  (N, num_classes) logits
        """
        # ①長さでソート（pack用）※enforce_sorted=Falseでも動くが、教科書式に明示
        lengths_sorted, perm_idx = lengths.sort(0, descending=True)
        tokens_sorted = tokens[perm_idx]

        # ②Embedding
        x = self.embedding(tokens_sorted)  # (N, T, embed_dim)

        # ③可変長をpackしてRNNへ（パッド部分を無視して計算）
        packed = pack_padded_sequence(x, lengths_sorted.cpu(), batch_first=True)

        # ④RNN
        # h_n: (num_layers * num_directions, N, hidden_size)
        _, h_n = self.rnn(packed)

        # ⑤ソートを元に戻す（h_n はソート順のままなので戻す）
        _, unperm_idx = perm_idx.sort(0)
        h_n = h_n[:, unperm_idx, :]

        # ⑥最終層の forward/backward の隠れ状態を合成して分類
        # num_layers>=1 のとき、最後の層の forward は -2、backward は -1 に入る
        h_fwd = h_n[-2]  # (N, hidden_size)
        h_bwd = h_n[-1]  # (N, hidden_size)
        h = h_fwd + h_bwd  # (N, hidden_size)  ※concatにするなら fcの入力次元も変える

        logits = self.fc(h)  # (N, num_classes)
        return logits


# ----------------------------
# 前処理：tokenize -> vocab -> ID化
# ----------------------------
def yield_tokens(data_list, tokenizer):
    for label, text in data_list:
        yield tokenizer(text)


def main(epochs=2, lr=0.05, momentum=0.9, batch_size=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # tokenizer
    tokenizer = get_tokenizer("basic_english")

    # Dataset（CIFAR10のdownload=True的に、自動DLされる）
    train_list = list(AG_NEWS(split="train"))
    test_list = list(AG_NEWS(split="test"))

    # vocab
    special_tokens = ["<unk>", "<pad>"]
    vocab = build_vocab_from_iterator(
        yield_tokens(train_list, tokenizer),
        specials=special_tokens,
        min_freq=2,
    )
    vocab.set_default_index(vocab["<unk>"])
    pad_idx = vocab["<pad>"]

    # pipeline
    def text_pipeline(text: str):
        return vocab(tokenizer(text))

    def label_pipeline(label: int):
        return int(label) - 1  # AG_NEWSは 1..4 なので 0..3 へ

    # DataLoader用 collate（padding + lengths）
    def collate_batch(batch):
        labels = []
        seqs = []
        lengths = []

        for (label, text) in batch:
            labels.append(label_pipeline(label))
            ids = torch.tensor(text_pipeline(text), dtype=torch.long)
            seqs.append(ids)
            lengths.append(ids.size(0))

        labels = torch.tensor(labels, dtype=torch.long)
        lengths = torch.tensor(lengths, dtype=torch.long)

        tokens_padded = pad_sequence(seqs, batch_first=True, padding_value=pad_idx)  # (N, T)
        return tokens_padded, lengths, labels

    trainloader = DataLoader(train_list, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    testloader = DataLoader(test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

    # Model / Loss / Optim
    net = BiRNN(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum)

    # ----------------------------
    # 学習
    # ----------------------------
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        for i, (tokens, lengths, labels) in enumerate(trainloader, 0):
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(tokens, lengths)     # (N, 4)
            loss = criterion(outputs, labels)  # labels: (N,)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 200 == 199:
                print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}")
                running_loss = 0.0

    print("Finished Training")

    # ----------------------------
    # 評価
    # ----------------------------
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, lengths, labels in testloader:
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
            outputs = net(tokens, lengths)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {100.0 * correct / total:.2f}%")
    return net


if __name__ == "__main__":
    main()


c:\Users\Nutzer\anaconda3\Lib\site-packages\torchtext\__init__.py:7: SyntaxWarning: invalid escape sequence '\ '
  "\n/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ \n"


OSError: Could not load this library: C:\Users\Nutzer\anaconda3\Lib\site-packages\torchtext\lib\libtorchtext.pyd

In [ ]:
class NetBiRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(
            embed_dim, hidden_size, num_layers=1, batch_first=True,
            bidirectional=True, nonlinearity="tanh"
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # 双方向なので 2H

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        # h_n: (layers*dirs, N, H) = (2, N, H) なので最後2つが forward/backward
        forward_last = h_n[-2]                # (N, H)
        backward_last = h_n[-1]               # (N, H)
        last_hidden = torch.cat([forward_last, backward_last], dim=1)  # (N, 2H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits
